# Week 9: Linear regression part 1

Last week you learned about principle component analysis using the SVD. This week you'll learn about the closely related method of linear regression. 

## Initialization cells

In [ ]:
import numpy as np
import numpy.linalg as la

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors

from sklearn.linear_model import LinearRegression

rng = np.random.default_rng()

## End of initialization cells

## The context: an explanation of linear regression using the SVD


You have a quantity $y$ and a collection of quantities $x^0,\ldots,x^{k-1}$, and you believe you have a linear relation

$$
 y = m_0 x^0 + \ldots + m_{k-1} x^{k-1} + b,
$$

but you don't know the coefficients $m_j$ or the constant term $b$. You'd like to estimate them. There could be many reasons for this: for example, you want to be able to make predictions about $y$ using the $x^j$.

Let's make vectors $m=[m_0,\ldots,m_{k-1}]$ and $x=[x^0,\ldots,x^{k-1}]$ so that in vector notation we have
$$
  y = m\cdot x + b,
$$
or in Python,
$$
y = m@x + b.
$$
So the challenge to estimate the vector $m$ and "constant term" $b$.

You have some measurements ("observations") of the quantity $y$ and the vector $x$: say $y_0,\ldots, y_{N-1}$ and $x_0,...,x_{N-1}$. But the data has some measurement error or other noise in it, so what you're really observing is

$$
 y_i = m @ x_i + b + \epsilon_i,
$$
where $\epsilon_i$ is an unknown error or noise in the $i$ measurement. 

How can you use this data to estimate $m$ and $b$?


As with the closely related method of PCA, it makes sense to turn our data into a matrix
$$
A = \begin{bmatrix} x_0 & 1 \\ \vdots & \vdots \\ x_{N-1} & 1 \end{bmatrix}
$$
Each row of $A$ has $k+1$ entries: in row $r$, the $0$ through $k-1$ entries are the entries of the observed $k$-dimensional vector $x_r$, and the last entry is the constant $1$. Let's also make a vector $v$ of unknowns out of our unknown $m$ and $b$,
$$
 v = \begin{bmatrix} m \\ b \end{bmatrix} = \begin{bmatrix} m_0 \\ m_1 \\ \vdots \\ m_{k-1} \\ b\end{bmatrix}.
$$
Then, using the dot product view of matrices acting on vectors, 
$$
A v = \begin{bmatrix} [x_0, 1] @ [m,b] \\ \vdots \\ [x_{N-1},1] @ [m,b] \end{bmatrix} = 
       \begin{bmatrix} m @ x_0 + b  \\ \vdots \\ m @  x_{N-1}+ b \end{bmatrix}.
$$

So if we let $Y$ be the vector

$$
Y = \begin{bmatrix}
y_0 \\
\vdots \\
y_{N-1} 
\end{bmatrix},
$$
then we're trying to find a $(k+1)$-dimension vector $v$ that solves the equation 
$$
Y = A v
$$

Now $A$ is an $N\times (k+1)$ matrix; it maps $k+1$-dimensional space to $N$-dimensional space. The image of $A$ is at most a $k+1$-dimensional plane in $\mathbb{R}^N$. If $N$ is larger than $k+1$, then we probably won't be able to solve this equation. Instead, we'll replace this problem with an approximation problem: 

>Linear regression problem: find $v=[m,b]$ so that $Av$ is as close as possible to $Y$.

The SVD tells us how to do this! If the trim SVD of $A$ is

$$
   A = H S C^T,
$$

then $H$ will have orthonormal columns of length $N$ [^1]

$$
 H = \begin{bmatrix} | & &  | \\
 h_0 & \ldots & h_{k} \\
 | & & |
 \end{bmatrix},
$$

$S$ will be the $(k+1)\times (k+1)$ diagonal matrix

$$
S = \mathrm{diag}([s_0,\ldots,s_{k}])
$$
and $C^T$ will have $k+1$ orthonormal rows of length $k+1$

$$
  C^T = \begin{bmatrix} -- & c_0 & --  \\ & \vdots & \\ -- & c_k & -- \end{bmatrix}.
$$


The image of $A$ is the span of the columns of $H$, and the closest you can get to $Y$ is the projection of $Y$ onto the span of the columns, which as we've seen is given by a dot product formula:

$$
   P_{H} (Y) = (h_0 \cdot Y) *  h_0 + \ldots +  (h_k\cdot Y) * h_k
$$
or in matrix notation
$$
  P_{H}(Y) =   H H^T Y \text{ or in Python }  H @ H.T @ Y.  
$$

We don't just want to find $P_{H}(Y)$. We want to find is the vector $v=[m,b]$ that solves the equation 

$$
   P_{H} (Y) = A @ v = H @ S@ C.T @ v.
$$   

From our study of the SVD, we know that 
$$
    A @ c_j = H @  S @ CT @ c_j = s_j * h_j,
$$

and applying matrices to vectors commutes with scalar multiplication, so to get $(h_j\cdot Y) * h_j$ we apply $A$ not to $c_j$ but to the scalar multiple $(h_j \cdot Y) * s_j^{-1}.$ This leads to the following conclusion:

>If 
>
$$
  v  =  s_0^{-1} * (h_0 \cdot Y) * c_0 +\ldots + s_k^{-1} * (h_k \cdot Y) * c_k
$$
>
>then $Av$ is the closest approximation to $Y$ possible using the data $A$.

There's a nice matrix formula for this $v$.  $v$ is a linear combination of the vectors $c_j$, so it should involve applying the matrix 

$$
 C = \begin{bmatrix}
     | & &| \\
     c_0 & \ldots &  c_k \\
     | &  &| 
\end{bmatrix}.
$$

Let $S^{-1}$ be the "inverse" of the matrix $S$: for a diagonal matrix this just means that 

$$
   S^{-1} = \mathrm{diag}([s_0^{-1},\ldots,s_k^{-1}])
$$

Notice that $S @ S^{-1}$ is the $(k+1)\times (k+1)$ identity matrix, the diagonal matrix with $1$s on the diagonal. 

Then the solution to the linear regression problem can be expressed as the following.

>**Fact:** If
>
$$
\begin{bmatrix}
m \\ b 
\end{bmatrix} = v = C S^{-1} H^T Y
$$
>or in Python `(C * (1/s)) @ H.T @ Y,`
>
>then $Av = P_{H}(Y)$: it is the closest approximation to $Y$ in the image of $A$.

Before turning to examples, let's see why. Since the $c_j$ are an orthonormal frame, 
we have

$$
   C^T  C = \begin{bmatrix} -- & c_0 & --  \\ & \vdots & \\ -- & c_k & -- \end{bmatrix}
   \begin{bmatrix}
     | & &| \\
     c_0 & \ldots &  c_k \\
     | &  &| 
\end{bmatrix} = I_{k+1}
$$

So multiplying by $C^T C$ doesn't change anything. So if $v= CS^{-1} H^T Y$, then

$$
A v = H S C^T C S^{-1} H^T Y = H S S^{-1}H^T Y = H H^T Y.
$$

Above we saw that $H H^T Y = P_{H} Y$, so 

$$
A v = P_{H} Y,
$$
which is what we were hoping for. 

[^1]: $H$ could have fewer column if $[x_0,\ldots,x_{N-1}]$ is a scalar multiple of $[1,\ldots,1]$. This is a special case of "colinearity of the right hand side variables", which is an important issue in linear regression models. 



## Linear regression: constructed example

We'll start with the simplest case: we're trying to model $y$ as $mx+b$, where $m$ and $b$ are unknown slopes. Let's pick random numbers for our slope and intercept $m$ and $b$.

In [ ]:
# rng.uniform returns uniformly distributed random numbers. let's make two between -3 and 3. 
m, b = rng.uniform(low=-3,high=3,size=(2))
# The goal isn't to keep these secret: we're going to try to recover them
print(f"m = {m:0.1f}")
print(f"b = {b:0.1f}")

We'll use our slope and constant term to make a function

In [ ]:
def f(x,m=None,b=None):
    return m * x + b

Now let's make a bunch of data: we'll make a vector of $x$'s and $y$'s by setting $y_j = m x_j + b + \epsilon_j$, where $\epsilon_j$ is some noise. 

In [ ]:
N = 25

X = rng.uniform(low=-3,high=3,size=N)

# We'll make the noise normally distributed with mean zero (the default for this random number generator)
# and standard deviation 0.5

noise = rng.normal(scale=0.5,size=N)

Y = f(X,m=m,b=b) + noise

fig,ax = plt.subplots()
ax.scatter(X,Y)

Let's use the method described in the "context" cell above to find the line that best fits these data.

In [ ]:
# As above, let A be the 2 by N matrix with the X values as its 0 column and ones as its 1 column
# the numpy function ones_like(X) returns an array of ones with the same shape as X.

A = np.array([X,np.ones_like(X)]).T

# A should be an N row by 2 column matrix. Let's check.
print("The shape of A is ", A.shape)
print(A)

In [ ]:

# Apply the SVD. Don't use the full matrices version

H, s, CT = la.svd(A,full_matrices=False)

# Let's look at the shape of these matrices

print("H should have two columns: in fact its shape is ",H.shape)
print("There should be two singular values: in fact s = ", s)
print("CT should be two-by-two: in fact its shape is ",CT.shape)


In [ ]:

# In the introduction text cell, we found that our estimates of m and b are given by C S^{-1} H^T Y
# CT is the transpose of C so to get C we transpose CT.
# In Python we can mutiply the 0 column of CT.T by s[0]^{-1} and the 1 column of CT.T by s[1]^{-1} by writing CT.T * (1/s).
    
mapprox, bapprox = (CT.T *(1./s)) @ H.T @ Y

print(f"The estimate of m from our data is {mapprox:0.2f}. The true value is {m:0.2f}.")
print(f"Our estimate of b is {bapprox:0.2f}. The true value is {b:0.2f}.")

#  Now plot the line that we've estimated to approximate our data 

fig,ax = plt.subplots()
ax.scatter(X,Y)
ax.plot(X,f(X,m=mapprox,b=bapprox),color='orange')


#### **Do it:** Try this yourself:
Repeat this process yourself, with $N=50$. Choose a new slope $m$ and a new intercept $b$.  Generate a vector of $N$ random numbers $X$ and a vector of random noise also of length $N$. Keep using `rng.normal` to generate your noise, but feel free to change the scale. Create a vector of observed $Y = m*X+b$.  Use  linear regression to obtain estimated values for $m$ and $b$, say `m_est` and `b_est`. Plot the points $(X,Y)$ (use `scatter`) and the line `y = m_est * x + b_est` (use `plot`).

In [ ]:
# you can start you work in this code cell

#### Leave this space for the grader

## Linear regression using sci-kit learn

It's valuable for you to see how linear regression is another application of the SVD and of representing data as a matrix. 

Still, linear regression is a very important tool, and of course many Python packages including scikit-learn provide you with tools to carry out linear regression without explicitly using the SVD.

Let's repeat our analysis above, using scikit-learn in place of the plain SVD.

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
# rng.uniform returns uniformly distributed random numbers. let's make two between -3 and 3. 
m, b = rng.uniform(low=-3,high=3,size=(2))

def f(x,m=None,b=None):
    return m * x + b

N = 25

X = rng.uniform(low=-1,high=1,size=N)

# We'll make the noise normally distributed with mean zero (the default for this random number generator)
# and standard deviation 0.7

noise = rng.normal(scale=0.7,size=N)

Y = f(X,m=m,b=b) + noise

fig,ax = plt.subplots()
ax.scatter(X,Y)

In [ ]:
model = LinearRegression()

# If Y has $N$ entries, then LinearRegresson wants X to be a N x 1 matrix instead of just an N-dimensional vector. 
# Notice that we didn't have to make the matrix $A$ with zero column $X$ and $1$ column a vector of $1$s.

model.fit(X.reshape(X.size,1),Y)

In [ ]:
# What model.predict does is to use the estimated slope $m$ and $b$ to predict the values of $y$ from 
# the values of $x$ that you pass to it. What plot does is to take the data you give it an draw lines between successive points
# The effect is to draw an orange line along where the X values are
# 


yfit = model.predict(X.reshape(X.size,1))
ax.plot(X,yfit,color='orange')

fig

In [ ]:
# You can also get the modeled value of m and b:
print(f"The actual value of m is {m:0.2f}. The modeled value is {model.coef_[0]:0.2f}.")
print(f"The actual value of b is {b:0.2f}. The modeled value is {model.intercept_:0.2f}.")

#### **Do it:** 
Try this yourself. Pick a slope m and intercept b (random is OK). Generate some random x values, and set $y$ using your slope and intercept plus some normally distributed noise. 

Then, use scikit-learn's LinearRegression model to fit the data. Plot the data and the fitted line. Find the fitted slope and intercept, and compare them to the slope and intercept that you started with. 

In [ ]:
# You can start your work in this code cell

#### Leave this cell for the grader

## Goodness of fit

There is so much to think about when you're building a statistical model! Recognizing the strengths and weaknesses of a model you've chosen is part of the art of statistics and data science.

This week we'll think about two measures of a model: $R^2$ and bias. Both are useful for many models, not just linear regression.

Let's focus first on $R^2$. The basic idea is to look at the sum of squares of the errors your model makes. We have a vector of observed $Y$s, and we have a vector of estimated values $Y_est.$.Our "errors" or "residuals" are the difference between the actually observed $Y$ and our estimate $Y_est$. The **sum of squared residuals** is

````
                SSResiduals = ((Y - Y_est)**2).sum() = (Y-Y_est) @ (Y-Y_est)
````

This quantity isn't very meaningful unless we adjust for how big $Y$ tends to be.  One way to do that is to look at how much $Y$ varies from its average. The **total sum of squares** of $Y$ is

````
                SSY= ((Y-Y.mean())**2).sum() = (Y-Y.mean()) @ (Y-Y.mean())
````

>**Definition:** The $R^2$ or **coefficient of determination** for our model is 
$$
 R^2 = 1 - \mathrm{SSResiduals}/\mathrm{SSY}.
$$

For example, if your model captures $Y$ precisely, then $Y=Y_est$ and so `SSResduals=0`. In that case $R^2 = 1$.
For example, perhaps the simplest estimate we could make for $Y$ is just to take its mean value: $Y_est$ is the constant vector 
$$
    Y_est = [Y.mean(), Y.mean(),\ldots,Y.mean()].
$$
In that case `SSResiduals = SSY`, and so $R^2 = 0$. Roughly speaking:

1. If your model fits your data perfectly then $R^2 = 1$
2. If your model performs better than always guessing the mean of $Y$, then $0<R^2\leq 1$.
3. If your model performs worse than always guessing the mean of $Y$, then $R^2 < 0.$

Roughly the $R^2$ is telling you how much of the variance in your $Y$ is captured by your model. People say "The model explains 100*R^2 percent of the variance in the data," although that's a funny thing to say when $R^2<0.$












In [ ]:
# For demonstration purposes, let's make a function Rsquared that returns this quatity for a vector of data $Y$ and 
# vector of modeled data Y_est

def R_squared(Y,Y_est):
    SSresiduals = (Y-Y_est)@(Y-Y_est)
    SSY = (Y-Y.mean()) @ (Y-Y.mean())
    return 1. - SSresiduals/SSY

Now let's make some data to examine $R^2$ with a linear model. We'll make a random linear model with a negative slope

$$
 y = m x + b
$$
with $m<0.$

We'll get a vector of random $X$, and our vector of observed $Y$ will be $m*X + b + \mathrm{noise}.$ We'll have `noise` be distributed have a bell curve distribution with mean $0$ and standard deviation $0.7.$

In [ ]:
# rng.uniform returns uniformly distributed random numbers. Let's get two random numbers, 
# and let's make sure that the slope $m$ is negative. 
m = rng.uniform(low=-3,high=0)
b = rng.uniform(low=-3,high=3)

def f(x,m=None,b=None):
    return m * x + b

N = 25

X = rng.uniform(low=-1,high=1,size=N)

# We'll make the noise normally distributed with mean zero (the default for this random number generator)
# and standard deviation 0.7

noise = rng.normal(scale=0.7,size=N)

Y = f(X,m=m,b=b) + noise

fig,ax = plt.subplots()
ax.scatter(X,Y)

For our first model, let's just always estimate the mean of $Y$.

In [ ]:
# sets Y_est to have the same shape as Y, with all its values the mean of $Y$.
Y_est = np.full_like(Y,Y.mean())

fig,ax = plt.subplots()
ax.scatter(X,Y,color='blue')
ax.plot(X,Y_est,color='orange')

print("The orange line is the constant line with y-value the mean of Y.")
print(f"For this model the R^2 is {R_squared(Y,Y_est):0.2f}. The model explains {100*R_squared(Y,Y_est):0.0f}% of the variance in the data.") 

Next let's use linear regression:

In [ ]:

# We follow the procedure above:
# 1. make a matrix A with our independent variable as its 0 column and a vector of 1s as its 1 column.

A = np.array([X,np.ones_like(X)]).T

# 2. run the SVD
H, s, CT = la.svd(A,full_matrices=False)

# 3. estimate the slope and intercept using the output of the SVD
m_est, b_est = (CT.T *(1./s)) @ H.T @ Y

# 4. Now compute Y_est
Y_est = f(X,m=m_est,b=b_est)

fig,ax = plt.subplots()
ax.scatter(X,Y,color='blue')
ax.plot(X,Y_est,color='orange')

print("The orange line is the line predicted by the linear regression.")
print(f"For this model the R^2 is {R_squared(Y,Y_est):0.2f}. The model explains {100*R_squared(Y,Y_est):0.0f}% of the variance in the data.") 


In [ ]:

# Now let's use a genuinely bad model with a positive slope: 

m_bad=0.5

Y_est = f(X,m=m_bad,b=b_est)

fig,ax = plt.subplots()
ax.scatter(X,Y,color='blue')
ax.plot(X,Y_est,color='orange')

print("The orange line is the line predicted by the linear regression.")
print(f"For this model the R^2 is {R_squared(Y,Y_est):0.2f}. The model explains {100*R_squared(Y,Y_est):0.0f}% of the variance in the data.") 


#### Do it: 

Try three variations of the experiment above:

First variation (baseline):

1. Choose $m$ and $b$ at random, between $-5$ and $5$, say
2. Set $N=25.$
3. Make a vector $X$ of $N$ random numbers between $-1$ and $1$.
4. Make a vector $noise$ of $N$ normally distributed random numbers with standard deviation $0.5$ 
5. Set `Y=m*X + b + noise`
6. Use linear regression on `Y` and `X` to estimate `m` and `b`: save those values as `m_est` and `b_est`.
7. Use your estimated `m_est` and `b_est` to estimate `Y` using `X`: save the estimated values as `Y_est`.
8. Plot the values `(X,Y)` and `(X,Y_est)`
9. Compute the `R^2`, save it as `Rsq1`

Second variation (Increase the size of the noise):

1. Keep $m$ and $b$ as in variation 1.
2. Keep $N$ as in variation 1.
3. Keep $X$ as in variation 1.
4. Make a new vector of noise called `noise2`, again normally distribued but now with standard deviation $1.5$.
5. Set `Y=m*X + b + noise2`
6. Use linear regression on `Y` and `X` to estimate $m$ and $b$ again; save those values as `m_2` and `b_2`.
7. Make your estimate `Y_est2` using `X`, `m_2`, and `b_2`
8. Plot as above
9. Compute the `R^2` for `Y` and `Y_est2`, and save it as `Rsq2`

Third variation (Keep the size of the noise; increase the size of the sample)

1. Keep $m$ and $b$ as in variation 1.
2. Now set $N=50$ instead of 25.
3. Make a new vector $X$ of $N$ random numbers between $-1$ and $1$.
4. Make a vector `noise` of $N$ normally distributed random numbers with standard deviation $0.5$  (same standard deviation as in variation 1)
5. Set `Y=m*X + b + noise`
6. Use linear regression on `Y` and `X` to estimate $m$ and $b$: save those values as `m_est` and `b_est`
7. Use your estimated `m_est` and `b_est` to estimate $Y$ using $X$: save the estimated values as `Y_est`
8. Plot the values $(X,Y)$ and `(X,Y_est)`
9. Compute the `R^2`, save it as `Rsq3`

In a text/markdown cell at the end of the notebook, discuss: 


In [ ]:
# use some code cells to do the variations

#### Use this markdown cell
for your answer to the question: How did the values $Rsq2$ and $Rsq3$ compare to $Rsq1$? Why do you think that is?

#### Grader: Leave this cell for the grader.